In [40]:
train = pd.read_csv("train.csv/train.csv")
test = pd.read_csv("test.csv/test.csv")

In [4]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", None)

In [5]:
train.head()

,Employee ID,Age,Gender,Years at Company,Job Role,Monthly Income,Work-Life Balance,Job Satisfaction,Performance Rating,Number of Promotions,Overtime,Distance from Home,Education Level,Marital Status,Number of Dependents,Job Level,Company Size,Company Tenure,Remote Work,Leadership Opportunities,Innovation Opportunities,Company Reputation,Employee Recognition,Attrition
0,8410,31,Male,19,Education,5390,Excellent,Medium,Average,2,No,22,Associate Degree,Married,0,Mid,Medium,89,No,No,No,Excellent,Medium,Stayed
1,64756,59,Female,4,Media,5534,Poor,High,Low,3,No,21,Master’s Degree,Divorced,3,Mid,Medium,21,No,No,No,Fair,Low,Stayed
2,30257,24,Female,10,Healthcare,8159,Good,High,Low,0,No,11,Bachelor’s Degree,Married,3,Mid,Medium,74,No,No,No,Poor,Low,Stayed
3,65791,36,Female,7,Education,3989,Good,High,High,1,No,27,High School,Single,2,Mid,Small,50,Yes,No,No,Good,Medium,Stayed
4,65026,56,Male,41,Education,4821,Fair,Very High,Average,0,Yes,71,High School,Divorced,0,Senior,Medium,68,No,No,No,Fair,Medium,Stayed


In [6]:
print("Train columns:", list(train.columns))
print("Test columns:", list(test.columns))

Train columns: ['Employee ID', 'Age', 'Gender', 'Years at Company', 'Job Role', 'Monthly Income', 'Work-Life Balance', 'Job Satisfaction', 'Performance Rating', 'Number of Promotions', 'Overtime', 'Distance from Home', 'Education Level', 'Marital Status', 'Number of Dependents', 'Job Level', 'Company Size', 'Company Tenure', 'Remote Work', 'Leadership Opportunities', 'Innovation Opportunities', 'Company Reputation', 'Employee Recognition', 'Attrition']
Test columns: ['Employee ID', 'Age', 'Gender', 'Years at Company', 'Job Role', 'Monthly Income', 'Work-Life Balance', 'Job Satisfaction', 'Performance Rating', 'Number of Promotions', 'Overtime', 'Distance from Home', 'Education Level', 'Marital Status', 'Number of Dependents', 'Job Level', 'Company Size', 'Company Tenure', 'Remote Work', 'Leadership Opportunities', 'Innovation Opportunities', 'Company Reputation', 'Employee Recognition', 'Attrition']


In [7]:
overlap = set(train["Employee ID"]) & set(test["Employee ID"])
print("Overlapping Employee IDs:", len(overlap))

Overlapping Employee IDs: 0


In [8]:
train["data_source"] = "train"
test["data_source"] = "test"

df = pd.concat([train, test], axis=0, ignore_index=True)

print("Combined shape:", df.shape)
print("Total rows:", len(df))
df["data_source"].value_counts()

Combined shape: (74498, 25)
Total rows: 74498


data_source
train    59598
test     14900
Name: count, dtype: int64

In [9]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace("-", "_")
    .str.replace(" ", "_")
)

df.columns.tolist()

['employee_id',
 'age',
 'gender',
 'years_at_company',
 'job_role',
 'monthly_income',
 'work_life_balance',
 'job_satisfaction',
 'performance_rating',
 'number_of_promotions',
 'overtime',
 'distance_from_home',
 'education_level',
 'marital_status',
 'number_of_dependents',
 'job_level',
 'company_size',
 'company_tenure',
 'remote_work',
 'leadership_opportunities',
 'innovation_opportunities',
 'company_reputation',
 'employee_recognition',
 'attrition',
 'data_source']

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 74498 entries, 0 to 74497
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   employee_id               74498 non-null  int64
 1   age                       74498 non-null  int64
 2   gender                    74498 non-null  str  
 3   years_at_company          74498 non-null  int64
 4   job_role                  74498 non-null  str  
 5   monthly_income            74498 non-null  int64
 6   work_life_balance         74498 non-null  str  
 7   job_satisfaction          74498 non-null  str  
 8   performance_rating        74498 non-null  str  
 9   number_of_promotions      74498 non-null  int64
 10  overtime                  74498 non-null  str  
 11  distance_from_home        74498 non-null  int64
 12  education_level           74498 non-null  str  
 13  marital_status            74498 non-null  str  
 14  number_of_dependents      74498 non-null  int64
 

In [11]:
missing = df.isnull().sum()
missing[missing > 0]

Series([], dtype: int64)

In [12]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate employee IDs:", df["employee_id"].duplicated().sum())

Duplicate rows: 0
Duplicate employee IDs: 0


In [13]:
df["attrition"].value_counts()

attrition
Stayed    39128
Left      35370
Name: count, dtype: int64

In [14]:
df["attrition"] = df["attrition"].map({"Stayed": 0, "Left": 1})

df["attrition"].value_counts()

attrition
0    39128
1    35370
Name: count, dtype: int64

In [15]:
yes_no_cols = [
    "overtime",
    "remote_work",
    "leadership_opportunities",
    "innovation_opportunities",
]

for col in yes_no_cols:
    df[col] = df[col].map({"Yes": 1, "No": 0})

df[yes_no_cols].head()

,overtime,remote_work,leadership_opportunities,innovation_opportunities
0,0,0,0,0
1,0,0,0,0
2,0,0,0,0
3,0,1,0,0
4,1,0,0,0


In [16]:
numeric_cols = [
    "age",
    "years_at_company",
    "monthly_income",
    "number_of_promotions",
    "distance_from_home",
    "number_of_dependents",
    "company_tenure",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[numeric_cols].describe()

,age,years_at_company,monthly_income,number_of_promotions,distance_from_home,number_of_dependents,company_tenure
count,74498.000000,74498.000000,74498.000000,74498.000000,74498.000000,74498.000000,74498.000000
mean,38.529746,15.721603,7299.379514,0.832935,49.991584,1.650326,55.727456
std,12.083456,11.223744,2152.508566,0.995289,28.513611,1.553633,25.399349
min,18.000000,1.000000,1226.000000,0.000000,1.000000,0.000000,2.000000
25%,28.000000,7.000000,5652.000000,0.000000,25.000000,0.000000,36.000000
50%,39.000000,13.000000,7348.000000,1.000000,50.000000,1.000000,56.000000
75%,49.000000,23.000000,8876.000000,2.000000,75.000000,3.000000,76.000000
max,59.000000,51.000000,16149.000000,4.000000,99.000000,6.000000,128.000000


In [17]:
print("Age range:", df["age"].min(), "to", df["age"].max())
print("Negative income rows:", (df["monthly_income"] < 0).sum())
print("Missing values after cleaning:", df.isnull().sum().sum())

Age range: 18 to 59
Negative income rows: 0
Missing values after cleaning: 0


In [18]:
total_employees = len(df)
left_count = df["attrition"].sum()
attrition_rate = left_count / total_employees * 100

print(f"Total employees: {total_employees:,}")
print(f"Left the company: {left_count:,}")
print(f"Attrition rate: {attrition_rate:.1f}%")

Total employees: 74,498
Left the company: 35,370
Attrition rate: 47.5%


In [19]:
role_attrition = (
    df.groupby("job_role")["attrition"]
    .mean()
    .sort_values(ascending=False)
    * 100
)

role_attrition.round(1)

job_role
Education     48.8
Healthcare    47.5
Technology    47.1
Finance       46.9
Media         46.8
Name: attrition, dtype: float64

In [20]:
gender_attrition = df.groupby("gender")["attrition"].mean() * 100
gender_attrition.round(1)

gender
Female    53.0
Male      42.9
Name: attrition, dtype: float64

In [21]:
income_by_attrition = df.groupby("attrition")["monthly_income"].mean()
income_by_attrition.index = ["Stayed", "Left"]
income_by_attrition.round(0)

Stayed    7321.0
Left      7275.0
Name: monthly_income, dtype: float64

In [22]:
overtime_attrition = df.groupby("overtime")["attrition"].mean() * 100
overtime_attrition.index = ["No Overtime", "Overtime"]
overtime_attrition.round(1)

No Overtime    45.5
Overtime       51.5
Name: attrition, dtype: float64

In [23]:
remote_attrition = df.groupby("remote_work")["attrition"].mean() * 100
remote_attrition.index = ["Not Remote", "Remote"]
remote_attrition.round(1)

Not Remote    52.8
Remote        24.7
Name: attrition, dtype: float64

In [24]:
df.groupby("attrition")[["age", "years_at_company", "distance_from_home"]].mean().round(1)

,age,years_at_company,distance_from_home
attrition,,,
0,39.1,16.4,47.4
1,37.9,14.9,52.8


In [25]:
attrition_labels = df["attrition"].map({0: "Stayed", 1: "Left"})
fig1 = px.pie(
    names=attrition_labels,
    title="Overall Employee Attrition",
    color=attrition_labels,
    color_discrete_map={"Stayed": "#2ecc71", "Left": "#e74c3c"},
)
fig1.show()

In [26]:
role_df = (
    df.groupby("job_role")["attrition"]
    .mean()
    .reset_index()
)
role_df["attrition_pct"] = role_df["attrition"] * 100

fig2 = px.bar(
    role_df.sort_values("attrition_pct", ascending=True),
    x="attrition_pct",
    y="job_role",
    orientation="h",
    title="Attrition Rate by Job Role (%)",
    labels={"attrition_pct": "Attrition Rate (%)", "job_role": "Job Role"},
    color="attrition_pct",
    color_continuous_scale="Reds",
)
fig2.show()

In [27]:
wlb_df = (
    df.groupby("work_life_balance")["attrition"]
    .mean()
    .reset_index()
)
wlb_df["attrition_pct"] = wlb_df["attrition"] * 100

fig3 = px.bar(
    wlb_df.sort_values("attrition_pct"),
    x="work_life_balance",
    y="attrition_pct",
    title="Attrition Rate by Work-Life Balance",
    labels={"attrition_pct": "Attrition Rate (%)", "work_life_balance": "Work-Life Balance"},
    color="attrition_pct",
    color_continuous_scale="Oranges",
)
fig3.show()

In [28]:
ot_df = df.groupby("overtime")["attrition"].mean().reset_index()
ot_df["status"] = ot_df["overtime"].map({0: "No Overtime", 1: "Overtime"})
ot_df["attrition_pct"] = ot_df["attrition"] * 100

fig5 = px.bar(
    ot_df,
    x="status",
    y="attrition_pct",
    title="Attrition Rate: Overtime vs No Overtime",
    labels={"attrition_pct": "Attrition Rate (%)", "status": ""},
    color="status",
    color_discrete_map={"No Overtime": "#3498db", "Overtime": "#e74c3c"},
)
fig5.show()

In [29]:
df.to_csv("cleaned_attrition_data.csv", index=False)
print("Saved cleaned_attrition_data.csv with", len(df), "rows")

Saved cleaned_attrition_data.csv with 74498 rows


Question 1

In [30]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv("cleaned_attrition_data.csv")

total = len(df)
left = df["attrition"].sum()
stayed = total - left
overall_rate = left / total * 100

print(f"Total Employees: {total:,}")
print(f"Left: {left:,}")
print(f"Stayed: {stayed:,}")
print(f"Overall Attrition Rate: {overall_rate:.1f}%")

role_attrition = (
    df.groupby("job_role")["attrition"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "attrition_rate", "count": "total"})
)
role_attrition["attrition_rate"] = (role_attrition["attrition_rate"] * 100).round(1)
role_attrition["left"] = (role_attrition["attrition_rate"] / 100 * role_attrition["total"]).astype(int)
role_attrition = role_attrition.sort_values("attrition_rate", ascending=False)

fig = px.bar(
    role_attrition.reset_index(),
    x="attrition_rate",
    y="job_role",
    orientation="h",
    text="attrition_rate",
    title="Attrition Rate by Job Role — Education Leads at 48.8%",
    labels={"attrition_rate": "Attrition Rate (%)", "job_role": "Job Role"},
    color_discrete_sequence=["#4C78A8"],
)
fig.add_vline(x=overall_rate, line_dash="dash", line_color="red", annotation_text=f"Company Avg: {overall_rate:.1f}%")
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(yaxis=dict(categoryorder="total ascending"), height=400)
fig.show()

print(f"\nInsight: Nearly half the workforce ({overall_rate:.1f}%) has left — this is a company-wide crisis, not a department problem.")
print(f"Education roles lead at {role_attrition.iloc[0]['attrition_rate']}% attrition ({role_attrition.iloc[0]['left']:,} people lost).")
print("However, all roles cluster within 2 percentage points, meaning the root causes are systemic.")
print("\nAction: Don't target one department — launch a company-wide retention program.")
print("Start with Education (highest rate) and Healthcare (largest headcount loss) as pilot groups.")

Total Employees: 74,498
Left: 35,370
Stayed: 39,128
Overall Attrition Rate: 47.5%



Insight: Nearly half the workforce (47.5%) has left — this is a company-wide crisis, not a department problem.
Education roles lead at 48.8% attrition (7,641.0 people lost).
However, all roles cluster within 2 percentage points, meaning the root causes are systemic.

Action: Don't target one department — launch a company-wide retention program.
Start with Education (highest rate) and Healthcare (largest headcount loss) as pilot groups.


Question 2

In [31]:
ot_attrition = df.groupby("overtime")["attrition"].mean() * 100
ot_counts = df.groupby("overtime")["attrition"].agg(["count", "sum"])
ot_counts.index = ["No Overtime", "Overtime"]
ot_counts.columns = ["Total", "Left"]

no_ot_rate = ot_attrition.iloc[0]
ot_rate = ot_attrition.iloc[1]
gap = ot_rate - no_ot_rate

print(f"No Overtime Attrition: {no_ot_rate:.1f}%")
print(f"Overtime Attrition: {ot_rate:.1f}%")
print(f"Gap: {gap:.1f} percentage points")

fig = go.Figure()
fig.add_trace(go.Bar(
    x=["No Overtime", "Overtime"],
    y=[no_ot_rate, ot_rate],
    text=[f"{no_ot_rate:.1f}%", f"{ot_rate:.1f}%"],
    textposition="outside",
    marker_color=["#4C78A8", "#E45756"],
    name="Attrition Rate"
))
fig.add_hline(y=overall_rate, line_dash="dash", line_color="gray",
              annotation_text=f"Company Avg: {overall_rate:.1f}%")
fig.update_layout(
    title=f"Overtime Employees Are {gap:.0f}pp More Likely to Leave",
    yaxis_title="Attrition Rate (%)",
    xaxis_title="",
    height=400,
    showlegend=False
)
fig.show()

print(f"\nInsight: Overtime workers leave at {ot_rate:.1f}% vs {no_ot_rate:.1f}% — a {gap:.1f}pp gap.")
print(f"That's roughly {gap/no_ot_rate*100:.0f}% higher relative risk.")
print("While overtime alone isn't the biggest driver, it amplifies burnout when combined with poor work-life balance.")
print("\nAction: HR should audit teams with chronic overtime (>3 months consecutive).")
print("Cap mandatory overtime and offer comp time or flex days to high-overtime teams.")

No Overtime Attrition: 45.5%
Overtime Attrition: 51.5%
Gap: 6.0 percentage points



Insight: Overtime workers leave at 51.5% vs 45.5% — a 6.0pp gap.
That's roughly 13% higher relative risk.
While overtime alone isn't the biggest driver, it amplifies burnout when combined with poor work-life balance.

Action: HR should audit teams with chronic overtime (>3 months consecutive).
Cap mandatory overtime and offer comp time or flex days to high-overtime teams.


Question 3

In [32]:
remote_counts = df["remote_work"].value_counts()
remote_pct = remote_counts / len(df) * 100

remote_attrition = df.groupby("remote_work")["attrition"].mean() * 100
onsite_rate = remote_attrition.iloc[0]
remote_rate = remote_attrition.iloc[1]
gap = onsite_rate - remote_rate

print(f"On-site employees: {remote_counts.iloc[0]:,} ({remote_pct.iloc[0]:.1f}%)")
print(f"Remote employees: {remote_counts.iloc[1]:,} ({remote_pct.iloc[1]:.1f}%)")
print(f"On-site attrition: {onsite_rate:.1f}%")
print(f"Remote attrition: {remote_rate:.1f}%")
print(f"Gap: {gap:.1f} percentage points")

fig = go.Figure()
fig.add_trace(go.Bar(
    x=["On-site", "Remote"],
    y=[onsite_rate, remote_rate],
    text=[f"{onsite_rate:.1f}%", f"{remote_rate:.1f}%"],
    textposition="outside",
    marker_color=["#E45756", "#4C78A8"],
    name="Attrition Rate"
))
fig.add_hline(y=overall_rate, line_dash="dash", line_color="gray",
              annotation_text=f"Company Avg: {overall_rate:.1f}%")
fig.update_layout(
    title=f"Remote Workers Leave {gap:.0f}pp Less Often — But Only {remote_pct.iloc[1]:.0f}% of Staff Are Remote",
    yaxis_title="Attrition Rate (%)",
    xaxis_title="",
    height=400,
    showlegend=False
)
fig.show()

print(f"\nInsight: Remote work shows the single largest attrition gap in the dataset: {gap:.1f}pp.")
print(f"However, only {remote_pct.iloc[1]:.0f}% of employees currently work remotely.")
print("This means: (a) the effect is real and large, but (b) it could reflect self-selection —")
print("remote workers may already be more engaged or in roles with higher autonomy.")
print("\nAction: Pilot a hybrid model for 2-3 high-attrition teams.")
print("Track attrition over 6 months to confirm whether the effect holds when expanded.")
print("Don't assume remote is a silver bullet — test before scaling.")

On-site employees: 60,300 (80.9%)
Remote employees: 14,198 (19.1%)
On-site attrition: 52.8%
Remote attrition: 24.7%
Gap: 28.1 percentage points



Insight: Remote work shows the single largest attrition gap in the dataset: 28.1pp.
However, only 19% of employees currently work remotely.
This means: (a) the effect is real and large, but (b) it could reflect self-selection —
remote workers may already be more engaged or in roles with higher autonomy.

Action: Pilot a hybrid model for 2-3 high-attrition teams.
Track attrition over 6 months to confirm whether the effect holds when expanded.
Don't assume remote is a silver bullet — test before scaling.


Question 4

In [33]:
results = []
for level in ["Entry", "Mid", "Senior"]:
    sub = df[df["job_level"] == level].copy()
    sub["income_quartile"] = pd.qcut(sub["monthly_income"], 4, labels=["Q1 (Lowest)", "Q2", "Q3", "Q4 (Highest)"])
    grp = sub.groupby("income_quartile")["attrition"].mean() * 100
    for q, rate in grp.items():
        results.append({"Job Level": level, "Income Quartile": q, "Attrition Rate (%)": round(rate, 1)})

pay_df = pd.DataFrame(results)

fig = px.bar(
    pay_df,
    x="Income Quartile",
    y="Attrition Rate (%)",
    color="Job Level",
    barmode="group",
    text="Attrition Rate (%)",
    title="Attrition by Pay Quartile Within Each Job Level — Level Matters More Than Pay",
    color_discrete_map={"Entry": "#E45756", "Mid": "#F58518", "Senior": "#4C78A8"},
    category_orders={"Income Quartile": ["Q1 (Lowest)", "Q2", "Q3", "Q4 (Highest)"],
                     "Job Level": ["Entry", "Mid", "Senior"]}
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(height=500, yaxis_range=[0, 75])
fig.show()

print("\nInsight: Job level is the dominant factor — Entry-level attrition (~63%) dwarfs Senior (~20%).")
print("Within each level, pay quartiles only shift attrition by 1-2pp.")
print("This means: paying more within the same band barely helps retention.")
print("The real 'pay fairness' issue is the gap between levels, not within them.")
print("\nAction: Don't raise salaries across the board — it won't move the needle.")
print("Instead, create faster promotion paths from Entry to Mid (where the 18pp drop happens).")
print("Review Entry-level pay bands only if they fall below market rate.")


Insight: Job level is the dominant factor — Entry-level attrition (~63%) dwarfs Senior (~20%).
Within each level, pay quartiles only shift attrition by 1-2pp.
This means: paying more within the same band barely helps retention.
The real 'pay fairness' issue is the gap between levels, not within them.

Action: Don't raise salaries across the board — it won't move the needle.
Instead, create faster promotion paths from Entry to Mid (where the 18pp drop happens).
Review Entry-level pay bands only if they fall below market rate.


Question 5

In [34]:
df["tenure_group"] = pd.cut(
    df["years_at_company"],
    bins=[0, 2, 5, 10, 20, 51],
    labels=["0–2 Years", "3–5 Years", "6–10 Years", "11–20 Years", "21+ Years"]
)

tenure_attrition = df.groupby("tenure_group")["attrition"].agg(["mean", "count"]).reset_index()
tenure_attrition["attrition_rate"] = (tenure_attrition["mean"] * 100).round(1)

fig = px.bar(
    tenure_attrition,
    x="tenure_group",
    y="attrition_rate",
    text="attrition_rate",
    title="Attrition Is Highest in the First 5 Years — Then Gradually Declines",
    labels={"tenure_group": "Years at Company", "attrition_rate": "Attrition Rate (%)"},
    color_discrete_sequence=["#4C78A8"],
)
fig.add_hline(y=overall_rate, line_dash="dash", line_color="red",
              annotation_text=f"Company Avg: {overall_rate:.1f}%")
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(
    height=450,
    xaxis=dict(categoryorder="array", categoryarray=["0–2 Years", "3–5 Years", "6–10 Years", "11–20 Years", "21+ Years"]),
    yaxis_range=[0, 60]
)
fig.show()

peak = tenure_attrition.iloc[0]["attrition_rate"]
late = tenure_attrition.iloc[-1]["attrition_rate"]

print(f"\nInsight: The first 5 years are the danger zone — attrition peaks at {peak}% for new hires (0–2 yrs).")
print(f"After 10 years, attrition drops to ~{late}% as employees become embedded.")
print("The biggest retention ROI comes from the first 5 years — not from retaining long-tenured staff.")
print("\nAction: Invest heavily in onboarding and the first-year experience.")
print("Assign mentors, schedule 30/60/90-day check-ins, and create an 'early career' growth track.")
print("Employees who survive year 5 are likely to stay — focus resources on getting them there.")


Insight: The first 5 years are the danger zone — attrition peaks at 52.9% for new hires (0–2 yrs).
After 10 years, attrition drops to ~43.7% as employees become embedded.
The biggest retention ROI comes from the first 5 years — not from retaining long-tenured staff.

Action: Invest heavily in onboarding and the first-year experience.
Assign mentors, schedule 30/60/90-day check-ins, and create an 'early career' growth track.
Employees who survive year 5 are likely to stay — focus resources on getting them there.


Question 6

In [35]:
combo = (
    df.groupby(["job_satisfaction", "work_life_balance"])["attrition"]
    .agg(["mean", "count"])
    .reset_index()
)
combo["attrition_rate"] = (combo["mean"] * 100).round(1)

sat_order = ["Low", "Medium", "High", "Very High"]
wlb_order = ["Poor", "Fair", "Good", "Excellent"]

pivot = combo.pivot(index="job_satisfaction", columns="work_life_balance", values="attrition_rate")
pivot = pivot.reindex(index=sat_order, columns=wlb_order)

fig = px.imshow(
    pivot,
    text_auto=".1f",
    color_continuous_scale="RdYlGn_r",
    title="Attrition Heatmap: Job Satisfaction × Work-Life Balance — Low Sat + Poor WLB = 67% Leave",
    labels={"x": "Work-Life Balance", "y": "Job Satisfaction", "color": "Attrition %"},
    aspect="auto"
)
fig.update_layout(height=450)
fig.show()

worst = combo.sort_values("mean", ascending=False).iloc[0]
print(f"\nInsight: The deadliest combination is {worst['job_satisfaction']} Satisfaction + {worst['work_life_balance']} Work-Life Balance")
print(f"at {worst['attrition_rate']}% attrition — {worst['attrition_rate'] - overall_rate:.1f}pp above company average.")
print(f"This group has {int(worst['count'])} employees — small but it's a canary in the coal mine.")
print("Any cell in the top-left (Low/Medium Satisfaction + Poor/Fair WLB) is a red flag.")
print("\nAction: Managers should watch for the combo, not individual scores.")
print("Flag any employee scoring both Low satisfaction AND Poor/Fair WLB in surveys.")
print("Trigger a 1-on-1 conversation within 2 weeks of a flagged survey result.")


Insight: The deadliest combination is Low Satisfaction + Poor Work-Life Balance
at 67.0% attrition — 19.5pp above company average.
This group has 994 employees — small but it's a canary in the coal mine.
Any cell in the top-left (Low/Medium Satisfaction + Poor/Fair WLB) is a red flag.

Action: Managers should watch for the combo, not individual scores.
Flag any employee scoring both Low satisfaction AND Poor/Fair WLB in surveys.
Trigger a 1-on-1 conversation within 2 weeks of a flagged survey result.


Question 7

In [36]:
df["age_group"] = pd.cut(df["age"], bins=[17, 25, 35, 45, 60], labels=["18–25", "26–35", "36–45", "46–60"])

life_stage = (
    df.groupby(["age_group", "marital_status"])["attrition"]
    .agg(["mean", "count"])
    .reset_index()
)
life_stage["attrition_rate"] = (life_stage["mean"] * 100).round(1)
life_stage = life_stage.sort_values("attrition_rate", ascending=False)

fig = px.bar(
    life_stage,
    x="age_group",
    y="attrition_rate",
    color="marital_status",
    barmode="group",
    text="attrition_rate",
    title="Young & Single Employees Leave at 72% — Life Stage Is a Major Attrition Driver",
    labels={"age_group": "Age Group", "attrition_rate": "Attrition Rate (%)", "marital_status": "Marital Status"},
    color_discrete_map={"Single": "#E45756", "Divorced": "#F58518", "Married": "#4C78A8"},
    category_orders={"age_group": ["18–25", "26–35", "36–45", "46–60"], "marital_status": ["Single", "Divorced", "Married"]}
)
fig.add_hline(y=overall_rate, line_dash="dash", line_color="gray",
              annotation_text=f"Company Avg: {overall_rate:.1f}%")
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(height=500, yaxis_range=[0, 82])
fig.show()

dep_attrition = df.groupby("number_of_dependents")["attrition"].mean() * 100

fig2 = px.bar(
    x=dep_attrition.index,
    y=dep_attrition.values,
    text=[f"{v:.1f}%" for v in dep_attrition.values],
    title="Employees With 4+ Dependents Leave Less — Family Ties Anchor Retention",
    labels={"x": "Number of Dependents", "y": "Attrition Rate (%)"},
    color_discrete_sequence=["#4C78A8"]
)
fig2.add_hline(y=overall_rate, line_dash="dash", line_color="red",
               annotation_text=f"Company Avg: {overall_rate:.1f}%")
fig2.update_traces(textposition="outside")
fig2.update_layout(height=400, yaxis_range=[0, 58])
fig2.show()

highest = life_stage.iloc[0]
print(f"\nInsight: The highest-risk life stage is {highest['age_group']}, {highest['marital_status']} at {highest['attrition_rate']}% attrition.")
print(f"That's {highest['attrition_rate'] - overall_rate:.1f}pp above average — these employees have the least to anchor them.")
print("Single employees across ALL age groups leave at 65-72%, far above married (36-42%).")
print("Dependents add a 'switching cost' — people with 4+ dependents leave 14pp less.")
print("\nAction: For young singles, focus on belonging and growth, not benefits they won't use.")
print("Offer career development programs, social events, and mentorship to build emotional ties.")
print("Don't offer family benefits to retain singles — offer purpose and progression.")


Insight: The highest-risk life stage is 18–25, Single at 72.0% attrition.
That's 24.5pp above average — these employees have the least to anchor them.
Single employees across ALL age groups leave at 65-72%, far above married (36-42%).
Dependents add a 'switching cost' — people with 4+ dependents leave 14pp less.

Action: For young singles, focus on belonging and growth, not benefits they won't use.
Offer career development programs, social events, and mentorship to build emotional ties.
Don't offer family benefits to retain singles — offer purpose and progression.


Question 8

In [37]:
promo_attrition = df.groupby("number_of_promotions")["attrition"].agg(["mean", "count"]).reset_index()
promo_attrition["attrition_rate"] = (promo_attrition["mean"] * 100).round(1)

fig = px.bar(
    promo_attrition,
    x="number_of_promotions",
    y="attrition_rate",
    text="attrition_rate",
    title="Promotions Dramatically Cut Attrition — 0 Promos = 49%, 3+ Promos = 25%",
    labels={"number_of_promotions": "Number of Promotions", "attrition_rate": "Attrition Rate (%)"},
    color_discrete_sequence=["#4C78A8"]
)
fig.add_hline(y=overall_rate, line_dash="dash", line_color="red",
              annotation_text=f"Company Avg: {overall_rate:.1f}%")
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(height=400, yaxis_range=[0, 58])
fig.show()

level_attrition = df.groupby("job_level")["attrition"].mean() * 100
level_order = ["Entry", "Mid", "Senior"]

fig2 = px.bar(
    x=level_order,
    y=[level_attrition[l] for l in level_order],
    text=[f"{level_attrition[l]:.1f}%" for l in level_order],
    title="Entry-Level Employees Leave at 3x the Rate of Seniors — Stagnation Kills Retention",
    labels={"x": "Job Level", "y": "Attrition Rate (%)"},
    color_discrete_sequence=["#4C78A8"]
)
fig2.update_traces(textposition="outside")
fig2.update_layout(height=400, xaxis=dict(categoryorder="array", categoryarray=level_order), yaxis_range=[0, 72])
fig2.show()

lead_yes = df[df["leadership_opportunities"] == 1]["attrition"].mean() * 100
lead_no = df[df["leadership_opportunities"] == 0]["attrition"].mean() * 100
innov_yes = df[df["innovation_opportunities"] == 1]["attrition"].mean() * 100
innov_no = df[df["innovation_opportunities"] == 0]["attrition"].mean() * 100

opps = pd.DataFrame({
    "Opportunity": ["Leadership", "Leadership", "Innovation", "Innovation"],
    "Status": ["No", "Yes", "No", "Yes"],
    "Attrition Rate (%)": [round(lead_no,1), round(lead_yes,1), round(innov_no,1), round(innov_yes,1)]
})

fig3 = px.bar(
    opps,
    x="Opportunity",
    y="Attrition Rate (%)",
    color="Status",
    barmode="group",
    text="Attrition Rate (%)",
    title="Leadership & Innovation Opportunities Reduce Attrition — But Few Get Them",
    color_discrete_map={"No": "#E45756", "Yes": "#4C78A8"}
)
fig3.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig3.update_layout(height=400, yaxis_range=[0, 55])
fig3.show()

lead_pct = df["leadership_opportunities"].mean() * 100
innov_pct = df["innovation_opportunities"].mean() * 100
zero_promo_pct = (df["number_of_promotions"] == 0).mean() * 100

print(f"\nInsight: Career stagnation is the clearest predictor of attrition in this dataset.")
print(f"- {zero_promo_pct:.0f}% of employees have ZERO promotions — they leave at 49%.")
print(f"- Only {lead_pct:.0f}% get leadership opportunities, only {innov_pct:.0f}% get innovation roles.")
print(f"- Entry-level employees leave at 63% vs Seniors at 20% — a 43pp gap.")
print(f"- 3+ promotions cuts attrition roughly in half (to ~25%).")
print("\nAction: Create lateral mobility and 'stretch assignment' programs for Entry/Mid levels.")
print("Set a target: every employee gets at least one growth opportunity per year.")
print("Fast-track high-performers from Entry to Mid within 18 months to close the gap.")


Insight: Career stagnation is the clearest predictor of attrition in this dataset.
- 50% of employees have ZERO promotions — they leave at 49%.
- Only 5% get leadership opportunities, only 16% get innovation roles.
- Entry-level employees leave at 63% vs Seniors at 20% — a 43pp gap.
- 3+ promotions cuts attrition roughly in half (to ~25%).

Action: Create lateral mobility and 'stretch assignment' programs for Entry/Mid levels.
Set a target: every employee gets at least one growth opportunity per year.
Fast-track high-performers from Entry to Mid within 18 months to close the gap.


Question 9

In [38]:
risk_profile = df[
    (df["overtime"] == 1) &
    (df["remote_work"] == 0) &
    (df["work_life_balance"] == "Poor") &
    (df["job_satisfaction"] == "Low")
]

risk_rate = risk_profile["attrition"].mean() * 100
risk_count = len(risk_profile)
risk_left = risk_profile["attrition"].sum()
multiplier = risk_rate / overall_rate

print("=" * 60)
print("HIGHEST-RISK EMPLOYEE PROFILE")
print("=" * 60)
print("Factors: Overtime + On-site + Poor Work-Life Balance + Low Job Satisfaction")
print(f"Attrition Rate: {risk_rate:.1f}%")
print(f"Company Average: {overall_rate:.1f}%")
print(f"Excess Risk: +{risk_rate - overall_rate:.1f} percentage points ({multiplier:.1f}x the average)")
print(f"Employees Matching: {risk_count} ({risk_count/len(df)*100:.2f}% of workforce)")
print(f"Already Left: {int(risk_left)}")

fig = go.Figure()
fig.add_trace(go.Bar(
    x=["Company Average", "High-Risk Profile"],
    y=[overall_rate, risk_rate],
    text=[f"{overall_rate:.1f}%", f"{risk_rate:.1f}%"],
    textposition="outside",
    marker_color=["#4C78A8", "#E45756"]
))
fig.update_layout(
    title=f"Highest-Risk Profile: {risk_rate:.0f}% Attrition — {multiplier:.1f}x Company Average ({risk_count} Employees)",
    yaxis_title="Attrition Rate (%)",
    xaxis_title="",
    height=400,
    showlegend=False,
    yaxis_range=[0, 90]
)
fig.show()

risk3 = df[
    (df["overtime"] == 1) &
    (df["remote_work"] == 0) &
    (df["work_life_balance"] == "Poor")
]
r3_rate = risk3["attrition"].mean() * 100
r3_count = len(risk3)

risk_alt = df[
    (df["overtime"] == 1) &
    (df["remote_work"] == 0) &
    (df["job_satisfaction"] == "Low")
]
ra_rate = risk_alt["attrition"].mean() * 100
ra_count = len(risk_alt)

print(f"\nAlternative 3-factor profiles:")
print(f"  OT + On-site + Poor WLB: {r3_rate:.1f}% attrition, {r3_count} employees")
print(f"  OT + On-site + Low Satisfaction: {ra_rate:.1f}% attrition, {ra_count} employees")

print(f"\nInsight: The 4-factor profile hits {risk_rate:.0f}% attrition but only covers {risk_count} people.")
print(f"The 3-factor version (OT + On-site + Poor WLB) still hits {r3_rate:.0f}% but catches {r3_count} employees.")
print("Both are actionable — the 3-factor version is better for scale.")
print("\nAction: Run a query on current employees matching 3+ factors.")
print("Proactively offer those employees schedule flexibility or remote days.")
print("This is a targeted intervention — small effort, high impact.")

HIGHEST-RISK EMPLOYEE PROFILE
Factors: Overtime + On-site + Poor Work-Life Balance + Low Job Satisfaction
Attrition Rate: 76.4%
Company Average: 47.5%
Excess Risk: +29.0 percentage points (1.6x the average)
Employees Matching: 259 (0.35% of workforce)
Already Left: 198



Alternative 3-factor profiles:
  OT + On-site + Poor WLB: 70.4% attrition, 2822 employees
  OT + On-site + Low Satisfaction: 62.5% attrition, 1951 employees

Insight: The 4-factor profile hits 76% attrition but only covers 259 people.
The 3-factor version (OT + On-site + Poor WLB) still hits 70% but catches 2822 employees.
Both are actionable — the 3-factor version is better for scale.

Action: Run a query on current employees matching 3+ factors.
Proactively offer those employees schedule flexibility or remote days.
This is a targeted intervention — small effort, high impact.


Question 10

In [39]:
drivers = {
    "Remote Work (On-site → Remote)": {
        "above_baseline": df[df["remote_work"] == 0]["attrition"].mean() * 100 - overall_rate,
        "gap": df[df["remote_work"] == 0]["attrition"].mean() * 100 - df[df["remote_work"] == 1]["attrition"].mean() * 100,
        "affected": (df["remote_work"] == 0).sum()
    },
    "Work-Life Balance (Poor → Good+)": {
        "above_baseline": df[df["work_life_balance"].isin(["Poor", "Fair"])]["attrition"].mean() * 100 - overall_rate,
        "gap": df[df["work_life_balance"].isin(["Poor", "Fair"])]["attrition"].mean() * 100 - df[df["work_life_balance"].isin(["Good", "Excellent"])]["attrition"].mean() * 100,
        "affected": df["work_life_balance"].isin(["Poor", "Fair"]).sum()
    },
    "Overtime Reduction": {
        "above_baseline": df[df["overtime"] == 1]["attrition"].mean() * 100 - overall_rate,
        "gap": df[df["overtime"] == 1]["attrition"].mean() * 100 - df[df["overtime"] == 0]["attrition"].mean() * 100,
        "affected": (df["overtime"] == 1).sum()
    }
}

ranking = sorted(drivers.items(), key=lambda x: x[1]["gap"] * x[1]["affected"], reverse=True)

print("=" * 70)
print("TOP 3 ATTRITION DRIVERS — RANKED BY POTENTIAL IMPACT")
print("=" * 70)

for rank, (name, vals) in enumerate(ranking, 1):
    potential_saves = int(vals["gap"] / 100 * vals["affected"] * 0.25)
    print(f"\n#{rank}: {name}")
    print(f"    Shift above baseline: +{vals['above_baseline']:.1f}pp")
    print(f"    Gap between groups: {vals['gap']:.1f}pp")
    print(f"    Employees affected: {vals['affected']:,}")
    print(f"    Estimated saves (25% gap closure): ~{potential_saves:,} employees")

top_name, top_vals = ranking[0]
potential = int(top_vals["gap"] / 100 * top_vals["affected"] * 0.25)

fig = go.Figure()
for rank, (name, vals) in enumerate(ranking):
    fig.add_trace(go.Bar(
        x=[name],
        y=[vals["gap"]],
        text=[f"{vals['gap']:.1f}pp"],
        textposition="outside",
        marker_color=["#E45756", "#F58518", "#4C78A8"][rank],
        name=name,
        showlegend=True
    ))
fig.update_layout(
    title="If HR Could Fix One Thing: Work-Life Balance Has the Widest Gap & Largest Reach",
    yaxis_title="Attrition Gap Between Groups (pp)",
    xaxis_title="",
    height=450,
    yaxis_range=[0, 30],
    showlegend=True
)
fig.show()

print(f"\n{'='*70}")
print(f"RECOMMENDATION: Fix #{ranking[0][0]}")
print(f"{'='*70}")
print(f"Why: It has the largest gap ({ranking[0][1]['gap']:.1f}pp) and affects {ranking[0][1]['affected']:,} employees.")
print(f"Estimated impact: Even a 25% improvement could retain ~{potential:,} employees per year.")
print(f"\nConcrete next step: Identify teams with the worst WLB/overtime scores.")
print("Pilot flexible scheduling and workload redistribution in those teams.")
print("Measure attrition at 6 months — expect a 5-8pp improvement in the pilot group.")

TOP 3 ATTRITION DRIVERS — RANKED BY POTENTIAL IMPACT

#1: Remote Work (On-site → Remote)
    Shift above baseline: +5.4pp
    Gap between groups: 28.1pp
    Employees affected: 60,300
    Estimated saves (25% gap closure): ~4,239 employees

#2: Work-Life Balance (Poor → Good+)
    Shift above baseline: +10.9pp
    Gap between groups: 19.5pp
    Employees affected: 32,908
    Estimated saves (25% gap closure): ~1,607 employees

#3: Overtime Reduction
    Shift above baseline: +4.0pp
    Gap between groups: 6.0pp
    Employees affected: 24,341
    Estimated saves (25% gap closure): ~362 employees



RECOMMENDATION: Fix #Remote Work (On-site → Remote)
Why: It has the largest gap (28.1pp) and affects 60,300 employees.
Estimated impact: Even a 25% improvement could retain ~4,239 employees per year.

Concrete next step: Identify teams with the worst WLB/overtime scores.
Pilot flexible scheduling and workload redistribution in those teams.
Measure attrition at 6 months — expect a 5-8pp improvement in the pilot group.
